# Regresión Logística
## Predicción de Mortalidad Hospitalaria - MIMIC-III

**Dataset:** MIMIC-III (Medical Information Mart for Intensive Care)  
**Fuente:** https://mimic.mit.edu/

## Variable objetivo (Y)
`hospital_expire_flag` — Si el paciente falleció en el hospital (1=Sí, 0=No)

## Features seleccionadas (n=10)

| # | Feature | Descripción | Justificación |
|---|---|---|---|
| 1 | `admission_type_encoded` | Tipo de admisión codificado | Emergencias tienen mayor riesgo |
| 2 | `age` | Edad del paciente en años | Pacientes mayores tienen mayor riesgo |
| 3 | `gender_encoded` | Género codificado (0=F, 1=M) | Factor demográfico relevante |
| 4 | `los_days` | Días de estancia hospitalaria | Estancias largas pueden indicar gravedad |
| 5 | `has_chartevents_data` | Si tiene datos de eventos | Indica monitoreo intensivo |
| 6 | `insurance_encoded` | Tipo de seguro codificado | Relacionado con acceso a atención |
| 7 | `admit_year` | Año de admisión | Captura tendencias temporales por año |
| 8 | `admit_month` | Mes de admisión | Captura estacionalidad |
| 9 | `admit_day` | Día del mes de admisión | Puede reflejar patrones operativos |
| 10 | `admit_dayofweek` | Día de la semana de admisión | Diferencias por carga asistencial semanal |

## Columnas descartadas

| Columna | Razón |
|---|---|
| `row_id`, `subject_id`, `hadm_id` | Identificadores — sin relación con mortalidad |
| `admittime`, `dischtime`, `deathtime` | Fechas — se usan para derivar features temporales y `los_days` |
| `edregtime`, `edouttime` | Tiempos de emergencia — muchos nulos |
| `diagnosis` | Texto libre — requiere NLP |
| `discharge_location` | Filtrado de información — contiene el outcome |

## Modelo a implementar
**Regresión Logística** para clasificación binaria

In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot

## 1. Carga del Dataset

Se cargan las tablas ADMISSIONS y PATIENTS.
Se combinan por subject_id para obtener información del paciente.
Se calculan features derivadas como edad y días de estancia.

In [8]:

# ── 1. Cargar los datos ──────────────────────────────────────────
admissions = pd.read_csv('ADMISSIONS.csv')
patients   = pd.read_csv('PATIENTS.csv')

# Pasar todos los nombres de columna a minúsculas
admissions.columns = admissions.columns.str.lower()
patients.columns   = patients.columns.str.lower()

# ── 2. Unir las dos tablas por paciente ──────────────────────────
data = admissions.merge(patients, on='subject_id', how='left')

# ── 3. Convertir fechas a formato fecha ──────────────────────────
for col in ['admittime', 'dischtime', 'dob']:
    data[col] = pd.to_datetime(data[col], errors='coerce')

# ── 4. Crear columnas nuevas (features) ─────────────────────────

# Edad del paciente al momento del ingreso
data['age'] = (data['admittime'] - data['dob']).dt.days / 365.25
data['age'] = data['age'].clip(0, 90)          # máximo 90 años
data['age'] = data['age'].fillna(data['age'].median())  # rellenar vacíos

# Días de estadía en el hospital
data['los_days'] = (data['dischtime'] - data['admittime']).dt.days
data['los_days'] = data['los_days'].fillna(0)

# Partes de la fecha de ingreso
data['admit_year']      = data['admittime'].dt.year
data['admit_month']     = data['admittime'].dt.month
data['admit_day']       = data['admittime'].dt.day
data['admit_dayofweek'] = data['admittime'].dt.dayofweek

# ── 5. Convertir categorías a números ───────────────────────────

# Género: 1 si es masculino, 0 si no
data['gender_encoded'] = (data['gender'].str.upper() == 'M').astype(int)

# Tipo de admisión: urgencia = 3, urgente = 2, electiva = 1, recién nacido = 0
tipo_admision = {'EMERGENCY': 3, 'URGENT': 2, 'ELECTIVE': 1, 'NEWBORN': 0}
data['admission_type_encoded'] = data['admission_type'].map(tipo_admision).fillna(0).astype(int)

# Seguro médico: asigna un número a cada tipo
data['insurance_encoded'] = data['insurance'].astype('category').cat.codes

# ── 6. Definir columnas de entrada (X) y objetivo (y) ───────────
FEATURES = [
    'admission_type_encoded',
    'age',
    'gender_encoded',
    'los_days',
    'has_chartevents_data',
    'insurance_encoded',
    'admit_year',
    'admit_month',
    'admit_day',
    'admit_dayofweek'
]
TARGET = 'hospital_expire_flag'   # 1 = falleció en el hospital, 0 = no

# ── 7. Preparar el dataset final ────────────────────────────────
df_final = data[FEATURES + [TARGET]].dropna()   # eliminar filas con vacíos

X = df_final[FEATURES].values.astype(np.float64)   # matriz de entrada
y = df_final[TARGET].values.astype(np.float64)     # vector objetivo
m = y.size                                          # número de filas

# ── 8. Resumen ───────────────────────────────────────────────────
print(f'Filas originales : {data.shape[0]:,}')
print(f'Filas limpias    : {df_final.shape[0]:,}')
print(f'Columnas (X)     : {len(FEATURES)}')
print(f'\nDistribución del target (falleció en hospital):')
print(df_final[TARGET].value_counts())

Filas originales : 129
Filas limpias    : 129
Columnas (X)     : 10

Distribución del target (falleció en hospital):
hospital_expire_flag
0    89
1    40
Name: count, dtype: int64


## 2. Exploración del Dataset

Se revisan estadísticas básicas para entender la distribución
de los datos antes de entrenar el modelo.

In [3]:
print('=== Información general ===')
data.info()
print()
print('=== Estadísticas básicas ===')
data[FEATURES + [TARGET]].describe()

=== Información general ===
<class 'pandas.DataFrame'>
RangeIndex: 129 entries, 0 to 128
Data columns (total 35 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   row_id_x                129 non-null    int64         
 1   subject_id              129 non-null    int64         
 2   hadm_id                 129 non-null    int64         
 3   admittime               129 non-null    datetime64[us]
 4   dischtime               129 non-null    datetime64[us]
 5   deathtime               40 non-null     str           
 6   admission_type          129 non-null    str           
 7   admission_location      129 non-null    str           
 8   discharge_location      129 non-null    str           
 9   insurance               129 non-null    str           
 10  language                81 non-null     str           
 11  religion                128 non-null    str           
 12  marital_status          113 non-n

,admission_type_encoded,age,gender_encoded,los_days,has_chartevents_data,insurance_encoded,admit_year,admit_month,admit_day,admit_dayofweek,hospital_expire_flag
count,129.000000,129.000000,129.000000,129.000000,129.000000,129.000000,129.000000,129.000000,129.000000,129.000000,129.000000
mean,2.860465,70.307552,0.542636,8.775194,0.992248,2.124031,2153.744186,6.821705,15.527132,2.751938,0.310078
std,0.495987,16.381486,0.500121,12.697342,0.088045,0.500121,30.657452,3.529894,8.776814,2.019557,0.464328
min,1.000000,17.190965,0.000000,0.000000,0.000000,0.000000,2102.000000,1.000000,1.000000,0.000000,0.000000
25%,3.000000,63.715264,0.000000,3.000000,1.000000,2.000000,2128.000000,4.000000,8.000000,1.000000,0.000000
50%,3.000000,72.728268,1.000000,6.000000,1.000000,2.000000,2150.000000,7.000000,15.000000,3.000000,0.000000
75%,3.000000,82.989733,1.000000,10.000000,1.000000,2.000000,2180.000000,10.000000,23.000000,5.000000,1.000000
max,3.000000,90.000000,1.000000,123.000000,1.000000,3.000000,2202.000000,12.000000,31.000000,6.000000,1.000000
